# Menstrual Cycle Data Preprocessing Pipeline
## Linear, Clean Data Preparation for Analysis

This notebook performs ALL preprocessing in a single linear flow:
1. Load raw data
2. Phase & bleeding detection
3. Flow volume imputation
4. Hormone processing (LH, Estrogen, PDG)
5. Symptom normalization
6. Final validation & export

**Output Fields (for explore_mcphases.ipynb):**

*Required:*
- `id`, `day_in_study`, `phase`, `study_interval`
- `lh`, `estrogen`, `pdg` (original, may have nulls)
- `flow_volume_imputed` (0-5 scale, no nulls)

*Imputed versions (filled, for model training):*
- `lh_imputed`, `estrogen_imputed`, `pdg_imputed` (same scales but all nulls filled)

*Derived:*
- `period_id`, `is_bleeding`, `flow_grp_*` (flow categories)
- Symptom columns mapped exactly as in `eda_cycle_pred.ipynb`


## Section 1: Setup & Load Data

In [51]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("Loading raw data from mcphases...")
hormones_selfrep = pd.read_csv(r'D:\MajorProject\mcphases\hormones_and_selfreport.csv')

print(f"Shape: {hormones_selfrep.shape}")
print(f"Participants: {hormones_selfrep['id'].nunique()}")
print(f"Days tracked: {hormones_selfrep['day_in_study'].max()}")
print(f"\nColumns: {hormones_selfrep.columns.tolist()}")

Loading raw data from mcphases...
Shape: (5659, 22)
Participants: 42
Days tracked: 1004

Columns: ['id', 'study_interval', 'is_weekend', 'day_in_study', 'phase', 'lh', 'estrogen', 'pdg', 'flow_volume', 'flow_color', 'appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings', 'indigestion', 'bloating']


## Section 2: Phase & Bleeding Features

In [52]:
# Fill missing phase values using forward fill
hormones_selfrep['phase'] = hormones_selfrep['phase'].fillna(method='ffill')

# Create binary bleeding indicator
hormones_selfrep['is_bleeding'] = (hormones_selfrep['phase'] == 'Menstrual').astype(int)

# Detect period starts (transition from non-menstrual to menstrual within same user)
hormones_selfrep = hormones_selfrep.sort_values(['id', 'day_in_study']).reset_index(drop=True)
hormones_selfrep['period_start'] = (
    (hormones_selfrep['phase'] == 'Menstrual') &
    (hormones_selfrep['phase'].shift(1) != 'Menstrual') &
    (hormones_selfrep['id'] == hormones_selfrep['id'].shift(1))
)

# Assign period IDs
hormones_selfrep['period_id'] = hormones_selfrep.groupby('id')['period_start'].cumsum()

# Calculate day within each period
hormones_selfrep['day_in_period'] = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
    .groupby(['id', 'period_id'])
    .cumcount() + 1
)
hormones_selfrep['day_in_period'] = hormones_selfrep['day_in_period'].fillna(0).astype(int)

print(f"Period start rows detected: {hormones_selfrep['period_start'].sum()}")
print(f"Cycles per person: {hormones_selfrep.groupby('id')['period_id'].nunique().describe()}")

Period start rows detected: 181
Cycles per person: count    42.000000
mean      5.309524
std       2.005944
min       2.000000
25%       4.000000
50%       4.000000
75%       7.000000
max      10.000000
Name: period_id, dtype: float64


C:\Users\iqra khan\AppData\Local\Temp\ipykernel_604\3420046283.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hormones_selfrep['phase'] = hormones_selfrep['phase'].fillna(method='ffill')


## Section 3: Flow Volume Processing

In [53]:
# Map categorical flow volume to numeric (0-5 scale)
flow_volume_map = {
    'Not at all': 0,
    'Spotting / Very Light': 1,
    'Light': 2,
    'Somewhat Light': 2.5,
    'Moderate': 3,
    'Somewhat Heavy': 3.5,
    'Heavy': 4,
    'Very Heavy': 5
}

hormones_selfrep['flow_volume_numeric'] = hormones_selfrep['flow_volume'].map(flow_volume_map)

print(f"Missing flow volume values: {hormones_selfrep['flow_volume_numeric'].isna().sum()}")
print(f"Distribution:\n{hormones_selfrep[hormones_selfrep['phase']=='Menstrual']['flow_volume_numeric'].value_counts().sort_index()}")

Missing flow volume values: 2470
Distribution:
flow_volume_numeric
0.0    126
1.0     59
2.0     86
2.5     77
3.0    119
3.5     62
4.0     55
5.0     24
Name: count, dtype: int64


In [54]:
# Build day-of-period flow profile from existing data
day_flow_profile = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
    .groupby('day_in_period')['flow_volume_numeric']
    .median()
)

population_mode = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
    ['flow_volume_numeric']
    .mode()[0] if len(hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']) > 0 else 2
)

print(f"Day-of-period flow profile:\n{day_flow_profile}")
print(f"Population mode: {population_mode}")

Day-of-period flow profile:
day_in_period
1     0.00
2     3.00
3     3.50
4     3.00
5     2.50
6     2.50
7     2.00
8     2.00
9     2.50
10    3.00
11    2.75
12    2.50
13     NaN
14    2.00
15     NaN
Name: flow_volume_numeric, dtype: float64
Population mode: 0.0


In [55]:
# Impute missing flow volumes using day profile + population mode
def impute_flow_volume(row):
    if pd.notna(row['flow_volume_numeric']):
        return row['flow_volume_numeric']
    if row['phase'] == 'Menstrual':
        return day_flow_profile.get(row['day_in_period'], population_mode)
    return 0

hormones_selfrep['flow_volume_imputed'] = hormones_selfrep.apply(impute_flow_volume, axis=1)

# Round to valid values to avoid fractional imputation artifacts
valid_values = [0, 1, 2, 2.5, 3, 3.5, 4, 5]
hormones_selfrep['flow_volume_imputed'] = (
    hormones_selfrep['flow_volume_imputed']
    .apply(lambda x: min(valid_values, key=lambda v: abs(x - v)))
)

print(f"Nulls after imputation: {hormones_selfrep['flow_volume_imputed'].isna().sum()}")
print(f"Non-menstrual with flow > 0: {len(hormones_selfrep[(hormones_selfrep['phase']!='Menstrual') & (hormones_selfrep['flow_volume_imputed']>0)])}")
print(f"\nFinal distribution:\n{hormones_selfrep['flow_volume_imputed'].value_counts().sort_index()}")

Nulls after imputation: 0
Non-menstrual with flow > 0: 245

Final distribution:
flow_volume_imputed
0.0    4539
1.0     198
2.0     187
2.5     218
3.0     294
3.5     142
4.0      56
5.0      25
Name: count, dtype: int64


## Section 4: Hormone Processing

In [56]:
# LH Surge Detection: LH > 2x baseline (follicular phase median)
user_lh_baseline = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Follicular']
    .groupby('id')['lh']
    .median()
)

def has_lh_surge(row):
    baseline = user_lh_baseline.get(row['id'], None)
    if baseline is None or pd.isna(row['lh']):
        return False
    return row['lh'] > (2 * baseline)

hormones_selfrep['lh_surge'] = hormones_selfrep.apply(has_lh_surge, axis=1)

print(f"LH surge days detected: {hormones_selfrep['lh_surge'].sum()}")
print(f"\nLH statistics:")
print(hormones_selfrep['lh'].describe())

LH surge days detected: 668

LH statistics:
count    5339.000000
mean        6.332609
std         7.888081
min         0.000000
25%         2.800000
50%         4.400000
75%         6.900000
max       185.600000
Name: lh, dtype: float64


In [57]:
# PDG (Progesterone Metabolite) Statistics & Imputation
PDG_THRESHOLD = 5.0

user_pdg_stats = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Luteal']
    .groupby('id')['pdg']
    .agg(['median', 'mean'])
)

population_luteal_pdg = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Luteal']
    ['pdg'].median()
)

population_follicular_pdg = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Follicular']
    ['pdg'].median()
)

print(f"PDG statistics:")
print(f"  Follicular median: {population_follicular_pdg}")
print(f"  Luteal median: {population_luteal_pdg}")
print(f"  Nulls: {hormones_selfrep['pdg'].isna().sum()}")

PDG statistics:
  Follicular median: 2.0
  Luteal median: 7.6
  Nulls: 3795


In [58]:
# Impute PDG based on phase logic
def impute_pdg(row):
    if pd.notna(row['pdg']):
        return row['pdg']
    
    user_median = user_pdg_stats.loc[row['id'], 'median'] if row['id'] in user_pdg_stats.index else None
    
    if row['phase'] == 'Luteal':
        return user_median if pd.notna(user_median) else population_luteal_pdg
    elif row['phase'] == 'Fertility':
        if row['lh_surge']:
            return PDG_THRESHOLD * 0.8  # Just after ovulation
        return population_follicular_pdg  # Pre-ovulation
    else:  # Follicular, Menstrual
        return population_follicular_pdg

hormones_selfrep['pdg_imputed'] = hormones_selfrep.apply(impute_pdg, axis=1)

print(f"PDG imputed (nulls remaining): {hormones_selfrep['pdg_imputed'].isna().sum()}")
print(f"PDG original nulls: {hormones_selfrep['pdg'].isna().sum()}")
print(f"PDG by phase after imputation:\n{hormones_selfrep.groupby('phase')['pdg_imputed'].median()}")

PDG imputed (nulls remaining): 0
PDG original nulls: 3795
PDG by phase after imputation:
phase
Fertility     2.0
Follicular    2.0
Luteal        7.6
Menstrual     2.0
Name: pdg_imputed, dtype: float64


In [59]:
# Impute LH & Estrogen with median (keep originals, create imputed versions)
hormones_selfrep['lh_imputed'] = hormones_selfrep['lh'].fillna(hormones_selfrep['lh'].median())
hormones_selfrep['estrogen_imputed'] = hormones_selfrep['estrogen'].fillna(hormones_selfrep['estrogen'].median())

print(f"LH original nulls: {hormones_selfrep['lh'].isna().sum()}")
print(f"LH imputed nulls: {hormones_selfrep['lh_imputed'].isna().sum()}")
print(f"Estrogen original nulls: {hormones_selfrep['estrogen'].isna().sum()}")
print(f"Estrogen imputed nulls: {hormones_selfrep['estrogen_imputed'].isna().sum()}")

LH original nulls: 320
LH imputed nulls: 0
Estrogen original nulls: 321
Estrogen imputed nulls: 0


## Section 5: Symptom Normalization

In [60]:
# Map symptom values exactly as in eda_cycle_pred.ipynb
severity_map = {
    "Not at all": 0,
    "Very Low/Little": 1,
    "Very Low"
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5
}

sym_cols = [
    'appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts',
    'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings',
    'indigestion', 'bloating'
]

for col in sym_cols:
    hormones_selfrep[col] = hormones_selfrep[col].map(severity_map)

In [61]:
# Check data types and value distribution exactly as in eda_cycle_pred.ipynb
symptom_cols = [
    'exerciselevel', 'appetite', 'sleepissue', 'foodcravings', 
    'indigestion', 'stress', 'bloating', 'moodswing', 'fatigue', 
    'headaches', 'sorebreasts', 'cramps'
]

for col in symptom_cols:
    print(f"\n{col}:")
    print(hormones_selfrep[col].value_counts(dropna=False).head(6))


exerciselevel:
exerciselevel
NaN    4155
3.0    1093
4.0     347
5.0      58
0.0       6
Name: count, dtype: int64

appetite:
appetite
NaN    3238
3.0    1748
4.0     580
5.0      89
0.0       4
Name: count, dtype: int64

sleepissue:
sleepissue
NaN    3097
3.0     688
0.0     662
1.0     649
4.0     362
5.0     201
Name: count, dtype: int64

foodcravings:
foodcravings
NaN    2974
0.0     922
3.0     624
1.0     595
4.0     391
5.0     153
Name: count, dtype: int64

indigestion:
indigestion
NaN    2903
0.0    1183
1.0     724
3.0     528
4.0     238
5.0      83
Name: count, dtype: int64

stress:
stress
NaN    2899
3.0    1050
4.0     610
0.0     436
1.0     370
5.0     290
Name: count, dtype: int64

bloating:
bloating
NaN    2889
0.0    1094
1.0     663
3.0     586
4.0     335
5.0      92
Name: count, dtype: int64

moodswing:
moodswing
NaN    2883
0.0    1032
1.0     780
3.0     612
4.0     268
5.0      84
Name: count, dtype: int64

fatigue:
fatigue
NaN    2880
3.0     936
4.0     685


## Section 7: Use Imputed Versions & Cleanup

In [62]:
# Drop temporary/redundant columns (keep BOTH original and imputed versions for transparency)
cols_to_drop = [
    'period_start',  # Already captured in period_id
    'flow_color',    # Replaced with flow_grp categories
    'flow_volume',   # Replaced with flow_volume_numeric and flow_volume_imputed
    'flow_volume_numeric',  # Replaced with flow_volume_imputed
    'lh_surge',      # Intermediate feature
    'day_in_period',  # Intermediate feature
]

cols_to_drop = [c for c in cols_to_drop if c in hormones_selfrep.columns]
hormones_selfrep = hormones_selfrep.drop(columns=cols_to_drop)

print(f"Dropped columns: {cols_to_drop}")
print(f"Kept original: pdg, lh, estrogen (may have nulls - shows missing data)")
print(f"Kept imputed: pdg_imputed, lh_imputed, estrogen_imputed (filled - ready for analysis)")
print(f"\nRemaining shape: {hormones_selfrep.shape}")

Dropped columns: ['period_start', 'flow_color', 'flow_volume', 'flow_volume_numeric', 'lh_surge', 'day_in_period']
Kept original: pdg, lh, estrogen (may have nulls - shows missing data)
Kept imputed: pdg_imputed, lh_imputed, estrogen_imputed (filled - ready for analysis)

Remaining shape: (5659, 26)


## Section 7: Column Cleanup & Selection

In [63]:
# Drop temporary/redundant columns
cols_to_drop = [
    'period_start',  # Already captured in period_id
    'flow_color',    # Replaced with flow_grp categories
    'flow_volume',   # Replaced with flow_volume_numeric and flow_volume_imputed
    'flow_volume_numeric',  # Replaced with flow_volume_imputed
    'lh_surge',      # Intermediate feature
    'day_in_period'  # Intermediate feature
]

cols_to_drop = [c for c in cols_to_drop if c in hormones_selfrep.columns]
hormones_selfrep = hormones_selfrep.drop(columns=cols_to_drop)

print(f"Dropped columns: {cols_to_drop}")
print(f"\nRemaining shape: {hormones_selfrep.shape}")

Dropped columns: []

Remaining shape: (5659, 26)


## Section 8: Data Validation

In [64]:
# Validate key columns for explore_mcphases compatibility
required_fields = ['id', 'day_in_study', 'phase', 'study_interval', 'lh', 'estrogen', 'pdg', 'flow_volume_imputed']
imputed_fields = ['pdg_imputed', 'lh_imputed', 'estrogen_imputed']

print("="*70)
print("VALIDATION REPORT FOR explore_mcphases.ipynb COMPATIBILITY")
print("="*70)

print("\n--- REQUIRED FIELDS ---")
for field in required_fields:
    if field in hormones_selfrep.columns:
        col = hormones_selfrep[field]
        print(f"\n✓ {field}")
        print(f"  dtype: {col.dtype}")
        print(f"  nulls: {col.isna().sum()}")
        if col.dtype in ['float64', 'int64']:
            print(f"  range: [{col.min():.2f}, {col.max():.2f}]")
        else:
            print(f"  unique: {col.nunique()} values")
    else:
        print(f"\n✗ {field} - MISSING!")

print("\n--- IMPUTED VERSIONS (for transparency) ---")
for field in imputed_fields:
    if field in hormones_selfrep.columns:
        col = hormones_selfrep[field]
        print(f"\n✓ {field}")
        print(f"  dtype: {col.dtype}")
        print(f"  nulls: {col.isna().sum()}")
        if col.dtype in ['float64', 'int64']:
            print(f"  range: [{col.min():.2f}, {col.max():.2f}]")

# Validate phase values
expected_phases = {'Menstrual', 'Follicular', 'Fertility', 'Luteal'}
actual_phases = set(hormones_selfrep['phase'].unique())
print(f"\nPhase validation:")
print(f"  Expected: {expected_phases}")
print(f"  Actual: {actual_phases}")
print(f"  Missing: {expected_phases - actual_phases}")
print(f"  Extra: {actual_phases - expected_phases}")

# Validate flow volume range
print(f"\nFlow volume imputed range validation:")
print(f"  Min: {hormones_selfrep['flow_volume_imputed'].min()}")
print(f"  Max: {hormones_selfrep['flow_volume_imputed'].max()}")
print(f"  Expected max: 5.0")

print(f"\n{len(hormones_selfrep):,} total rows ready for analysis")

VALIDATION REPORT FOR explore_mcphases.ipynb COMPATIBILITY

--- REQUIRED FIELDS ---

✓ id
  dtype: int64
  nulls: 0
  range: [1.00, 50.00]

✓ day_in_study
  dtype: int64
  nulls: 0
  range: [1.00, 1004.00]

✓ phase
  dtype: object
  nulls: 0
  unique: 4 values

✓ study_interval
  dtype: int64
  nulls: 0
  range: [2022.00, 2024.00]

✓ lh
  dtype: float64
  nulls: 320
  range: [0.00, 185.60]

✓ estrogen
  dtype: float64
  nulls: 321
  range: [0.00, 640.00]

✓ pdg
  dtype: float64
  nulls: 3795
  range: [1.00, 30.00]

✓ flow_volume_imputed
  dtype: float64
  nulls: 0
  range: [0.00, 5.00]

--- IMPUTED VERSIONS (for transparency) ---

✓ pdg_imputed
  dtype: float64
  nulls: 0
  range: [1.00, 30.00]

✓ lh_imputed
  dtype: float64
  nulls: 0
  range: [0.00, 185.60]

✓ estrogen_imputed
  dtype: float64
  nulls: 0
  range: [0.00, 640.00]

Phase validation:
  Expected: {'Fertility', 'Follicular', 'Luteal', 'Menstrual'}
  Actual: {'Fertility', 'Follicular', 'Menstrual', 'Luteal'}
  Missing: set(

## Section 9: Export Processed Data

In [65]:
import os

# Export to CSV for explore_mcphases.ipynb
output_path = r'D:\MajorProject\EDA\hormones_selfrep_processed.csv'
hormones_selfrep.to_csv(output_path, index=False)

print(f"✓ Saved to: {output_path}")
print(f"\nFile info:")
print(f"  Rows: {len(hormones_selfrep):,}")
print(f"  Columns: {hormones_selfrep.shape[1]}")
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"  File size: {np.round(file_size_mb, 2)} MB")

print(f"\nDataframe info:")
print(hormones_selfrep.info())

✓ Saved to: D:\MajorProject\EDA\hormones_selfrep_processed.csv

File info:
  Rows: 5,659
  Columns: 26
  File size: 0.48 MB

Dataframe info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5659 entries, 0 to 5658
Data columns (total 26 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   5659 non-null   int64  
 1   study_interval       5659 non-null   int64  
 2   is_weekend           5659 non-null   bool   
 3   day_in_study         5659 non-null   int64  
 4   phase                5659 non-null   object 
 5   lh                   5339 non-null   float64
 6   estrogen             5338 non-null   float64
 7   pdg                  1864 non-null   float64
 8   appetite             2421 non-null   float64
 9   exerciselevel        1504 non-null   float64
 10  headaches            2824 non-null   float64
 11  cramps               3032 non-null   float64
 12  sorebreasts          2977 non-null   float64
 1

## Section 10: Quick Preview

In [66]:
# Display sample of processed data
print("\nSample rows from processed dataset:")
display(hormones_selfrep.head(10))

print("\nDescriptive statistics:")
display(hormones_selfrep.describe())


Sample rows from processed dataset:


,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,appetite,exerciselevel,...,stress,foodcravings,indigestion,bloating,is_bleeding,period_id,flow_volume_imputed,pdg_imputed,lh_imputed,estrogen_imputed
0,1,2022,True,1,Follicular,2.9,94.2,NaN,NaN,NaN,...,3.0,1.0,1.0,1.0,0,0,0.0,2.0,2.9,94.2
1,1,2022,False,2,Follicular,1.2,226.3,NaN,NaN,NaN,...,3.0,1.0,1.0,1.0,0,0,0.0,2.0,1.2,226.3
2,1,2022,False,3,Follicular,3.5,276.8,NaN,NaN,NaN,...,NaN,1.0,1.0,1.0,0,0,0.0,2.0,3.5,276.8
3,1,2022,False,4,Fertility,1.8,322.1,NaN,NaN,NaN,...,NaN,1.0,1.0,1.0,0,0,0.0,2.0,1.8,322.1
4,1,2022,False,5,Fertility,4.6,244.9,NaN,NaN,NaN,...,NaN,1.0,1.0,1.0,0,0,0.0,2.0,4.6,244.9
5,1,2022,False,6,Fertility,5.0,364.7,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,0,0,0.0,2.0,5.0,364.7
6,1,2022,True,7,Fertility,5.0,364.7,NaN,3.0,3.0,...,NaN,1.0,1.0,1.0,0,0,0.0,2.0,5.0,364.7
7,1,2022,True,8,Fertility,5.0,364.7,NaN,3.0,3.0,...,NaN,1.0,1.0,1.0,0,0,0.0,2.0,5.0,364.7
8,1,2022,False,9,Fertility,4.6,222.3,NaN,NaN,NaN,...,1.0,1.0,1.0,1.0,0,0,0.0,2.0,4.6,222.3
9,1,2022,False,10,Luteal,3.0,63.6,NaN,NaN,NaN,...,3.0,1.0,1.0,1.0,0,0,0.0,7.6,3.0,63.6



Descriptive statistics:


,id,study_interval,day_in_study,lh,estrogen,pdg,appetite,exerciselevel,headaches,cramps,...,stress,foodcravings,indigestion,bloating,is_bleeding,period_id,flow_volume_imputed,pdg_imputed,lh_imputed,estrogen_imputed
count,5659.000000,5659.000000,5659.000000,5339.000000,5338.000000,1864.000000,2421.000000,1504.000000,2824.000000,3032.000000,...,2760.000000,2685.000000,2756.000000,2770.000000,5659.000000,5659.000000,5659.000000,5659.000000,5659.000000,5659.000000
mean,26.545503,2022.693055,347.474289,6.332609,128.987748,6.229828,3.308137,3.295878,1.367210,0.907652,...,2.687681,1.786220,1.333454,1.523827,0.190670,2.522707,0.502739,4.935810,6.223326,127.346289
std,14.618883,0.951811,416.521099,7.888081,101.859259,7.112508,0.553774,0.577543,1.507356,1.288753,...,1.582431,1.696548,1.531112,1.608587,0.392864,1.923687,1.095050,5.345072,7.674766,99.153877
min,1.000000,2022.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,13.000000,2022.000000,34.000000,2.800000,67.100000,1.500000,3.000000,3.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,2.000000,2.900000,68.900000
50%,26.000000,2022.000000,69.000000,4.400000,100.050000,3.400000,3.000000,3.000000,1.000000,0.000000,...,3.000000,1.000000,1.000000,1.000000,0.000000,2.000000,0.000000,2.000000,4.400000,100.050000
75%,41.000000,2024.000000,893.000000,6.900000,155.400000,7.700000,4.000000,4.000000,3.000000,1.000000,...,4.000000,3.000000,3.000000,3.000000,0.000000,4.000000,0.000000,7.600000,6.650000,151.100000
max,50.000000,2024.000000,1004.000000,185.600000,640.000000,30.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,5.000000,1.000000,9.000000,5.000000,30.000000,185.600000,640.000000
